# D1 — MAE vs MFE Scatter by Exit Type

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
EXIT_COLORS = {'TP': '#2ecc71', 'SL': '#e74c3c', 'EOD': '#95a5a6', 'BE': '#f39c12'}
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (wk, wm) in zip(axes, WINDOWS_META.items()):
    rk = wm['result_key']
    trades = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'trades.parquet')
    for etype, grp in trades.groupby('exit_type'):
        c = EXIT_COLORS.get(etype, 'grey')
        ax.scatter(grp['mae_pts'], grp['mfe_pts'], color=c, alpha=0.7, s=30, label=etype)
    ax.set_xlabel('MAE (pts)')
    ax.set_ylabel('MFE (pts)')
    ax.set_title(wk)
    ax.axhline(0, color='black', lw=0.4)
    ax.axvline(0, color='black', lw=0.4)
    handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=l)
               for l, c in EXIT_COLORS.items()]
    ax.legend(handles=handles, fontsize=7)
# annotate W3 trade 27 (worst spike loss)
ax_w3 = axes[2]
wm3 = WINDOWS_META['W3']
t3 = pd.read_parquet(RESULTS_DIR / wm3['result_key'] / 'none' / 'trades.parquet')
worst = t3.loc[t3['mae_pts'].idxmax()]
ax_w3.annotate('Trade 27\n−$218.75', xy=(worst['mae_pts'], worst['mfe_pts']),
               xytext=(worst['mae_pts']-0.3, worst['mfe_pts']+0.5),
               fontsize=7, arrowprops=dict(arrowstyle='->', color='black'))
fig.suptitle('MAE vs MFE by Exit Type — All Windows', fontsize=13)
save_fig(fig, 'D1_mae_mfe_scatter.png')
